BIDIRECTINAL RNNs

La Bidirectional RNN (BRNN) nasce per risolvere uno dei limiti più importanti delle RNN, delle GRU e delle LSTM tradizionali.
Una LSTM standard legge il testo solo da sinistra verso destra
Ma quando noi leggiamo una frase, spesso comprendiao il significato di una parola anche grazie a ciò che viene dopo.
Questa è l'idea alla base della Bidirectional RNN

La soluzione è semplice
Perchè usare una sola LSTM?
usiamono due
la prima legge da sinistra verso destra
la seconda legge da destra verso sinistra.

Per ogni parola vengono prodotti due stati nascosti.
Uno proveniente dalla lettura normale
Uno proveniente dalla lettura inversa.
Le due reti lavorano contemporaneamente
Per ogni parola otteniamo:  - forward hidden state - backward hidden state
Supponiamo hidden_size=64
forward produce 64 valori
backwaord produce 64 valori, alla fine abbiamo 128 valori

Concaenzazione degli Stati
Unire i due mondi (da sinistra verso destra e da destra verso sinistra) informativi
Una volta che i due rami hanno terminato l'elaborazione, i loro output devono essere combinati per produrre una singola previsione per ogni istante temporale.
Dobbiamo decidere come combiare le due informazioni.
La concatenazione è il metodo standard che preserva l'integrità delle informazioni provenienti da entrambe le direzioni senza perdita di dettaglio.

Fisacamente l'output al tempo t diventa un vettore lungo il doppio rispetto a quello di una RNN unidirezionale. 
I due vettori hidden vengono affiancati, mantenendo separate le feature temporali.
In Kers e PyTorch tutto questo avviene in modo quasi invisibile, ma è fondamentale campire che stiamo aumentando la dimensionalità del nostro spazio delle feature.
Stiamo letteralmente addestrando due reti, richiede più memoria e tempo, tuttavia il guadagno in termini di precisione è così elevato da giustificare questo sovraprezzo computazionale.

Utilità nel Riconoscimento Vocale e Traduzione

Nel linguaggio naturale, una parola può essere ambigua finchè non leggiamo il resto della frase. Le BRNN eccellono proprio nel risolvere queste ambiguità
Questa capacità di 'guardare avanti' rende più precisi i sistemi di trascrizione e traduzione automatica.
Casi d'uso
- Riconoscimento Vocale: i fonemi iniziali vengono corretti basandosi sulla finel della parola o della frase pronunciata. Se dico una parola che suona come un'altra il sistema aspetta la fine della frase per correggere la trascrizione iniziale in base al contesto.
- Traduzione automatica: catturare la struttura sintattica di lingue con ordine e parole differente prima di generare l'output
- Tagging delle parti del discorso: capire se una parola è un verbo o un sostantivo osservando i modificatori successivi
- Protein Sequencing: in bioinformatica, per analizzare sequenze dove la funzione di un amminoacido dipende dai vicini in entrambe le direzioni

Limiti e alternative moderne
- Latenza del sistema: poiche dobbiamo attendere la fine della sequenza, le BRNN non sono ideali per la traduzione simultanea 'live' istantanea.
- Confronto con i Transformer: Sebbene i Transformer stiano sostituendo le BRNN in molti campi, queste rimangono una soluzione solida e meno esigente in termini di dati per sequenze medie. I transformer gestiscono il contesto globale in  modo ancorra più efficiente su sequenze molto lunghe.
- Stacking bidirezionale: è possibile impilare più layer dibirezionali, dove l'output concatenato di un livello diventa l'input per i livelli bidirezionale successivo.

In PyTorch basta aggiungere bidirectional
self.lstm = nn.LSTM(
    input_size=128,
    hidden_size=64,
    bidirectional=True,
    batch_first=True
)
In Keras è ancora più semplice:
non dobbiamo costruire i due rami in modo manuale.
model.add(layers.Bidirectional(layers.LSTM(64)))
Questo rende il passaggio da un modello unidirezionale a uno bidirezionale estremamente rapido a livello di codice.
model = Sequential([
    Embedding(...),

    Bidirectional(
        LSTM(64)
    ),

    Dense(1, activation="sigmoid")
])

Le Bidirectional RNN hanno avuto un enorme successo nei primi sistemi di traduzione automatica.

LIMITI

1. Non può lavorare realmente in tempo reale.
Per leggere all'indietro bisogna conoscere tutta la frase.
Se stai ricevendo un testo parola per parola:
ciao come stai
non puoi partire dalla fine perchè... la fine non esiste ancora.
Per questo motico le bidirectional RNN non sono adatte a scenare di straming puro.
2. Sono più lente
Utilizzano due reti invece di una, circa il doppio dei calcoli.
3. Più Memoria
Anche glistati nascosti raddoppiano

PERCHE' OGGI SI USANO MENO
Dal 2017 con il lavoro 'Attention Is All You Need' i Transformer hanno progressivamente sostituito le RNN nella maggior parte delle applicazioni NPL.
Modelli come:
- Bert
- RoBerta
- DistilBERT
sono bidirezionali in modo più efficace
La differenza è importante:
- una Bidirectional LSTM guarda il passato e il futuro in modo sequenziale.
- un Transformer usa il meccanismo dle self-attention che permette a ogni parola di 'guardare' direttamente tutte le altre parole della frese nello stesso momento.
Per questo motivo BERT comprende relazioni molto lontane nel testo con maggiore efficacia rispetto a una Bidirectional LSTM




In [1]:
import os
os.environ["KERAS_BACKEND"] = "torch"

import numpy as np
import keras
from keras import layers

# 1. GENERAZIONE DATI SINTETICI (Simuliamo una sequenza di testo)
# Immaginiamo 1000 campioni, ogni sequenza lunga 10, con 8 feature per step
num_samples = 1000
seq_len = 10
embedding_dim = 8

X = np.random.rand(num_samples, seq_len, embedding_dim).astype("float32")
y = np.random.randint(0, 2, size=(num_samples, 1)) # Classificazione binaria

# 2. MODELLO UNIDIREZIONALE (Baseline)
# Teoria: La RNN elabora i dati solo da t=0 a t=end. 
# Lo stato finale cattura solo ciò che è accaduto "prima".
model_uni = keras.Sequential([
    layers.Input(shape=(seq_len, embedding_dim)),
    layers.LSTM(32), # 32 unità hidden
    layers.Dense(1, activation='sigmoid')
], name="Unidirectional_Model")

# 3. MODELLO BIDIREZIONALE
# Teoria: Usiamo il wrapper 'Bidirectional'. Keras creerà internamente 
# DUE istanze di LSTM: una per il flusso forward e una per quello backward.
model_bi = keras.Sequential([
    layers.Input(shape=(seq_len, embedding_dim)),
    # merge_mode='concat' è il default: concatena gli stati (32 + 32 = 64 feature)
    layers.Bidirectional(layers.LSTM(32), merge_mode='concat'),
    layers.Dense(1, activation='sigmoid')
], name="Bidirectional_Model")

# 4. ANALISI E CONFRONTO
print("--- Riepilogo Modello Unidirezionale ---")
model_uni.summary() 
# Notate l'output del layer LSTM: (None, 32)

print("\n--- Riepilogo Modello Bidirezionale ---")
model_bi.summary()
# Teoria: L'output qui è (None, 64). 
# Questo perché abbiamo concatenato lo stato forward (32) e quello backward (32).
# Il numero di parametri è quasi il doppio perché abbiamo letteralmente due LSTM.

# Compilazione e test rapido
model_bi.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model_bi.fit(X, y, epochs=1, batch_size=32, verbose=0)
print("\nTraining completato con successo.")


def compare_accuracy(model1, model2, data_X, data_y, epochs=10):
    """
    Addestra e confronta le performance di due modelli.
    Teoria: Misuriamo quanto velocemente la 'Loss' scende e l'Accuracy sale.
    """
    print(f"\n Inizio addestramento comparativo per {epochs} epoche...")
    
    # Compilazione: Settiamo le regole del gioco (Ottimizzatore e Funzione di Errore)
    model1.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    model2.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    
    # Training: La fase in cui i pesi W e V vengono aggiornati tramite Backpropagation
    print(f"Addestramento {model1.name}...")
    history_uni = model1.fit(data_X, data_y, epochs=epochs, batch_size=32, verbose=0, validation_split=0.2)
    
    print(f"Addestramento {model2.name}...")
    history_bi = model2.fit(data_X, data_y, epochs=epochs, batch_size=32, verbose=0, validation_split=0.2)
    
    # Estrazione Risultati Finali
    acc_uni = history_uni.history['val_accuracy'][-1]
    acc_bi = history_bi.history['val_accuracy'][-1]
    
    print("\n" + "="*40)
    print("RISULTATI FINALI (Dati di Validazione)")
    print("="*40)
    print(f"Acc. Unidirezionale: {acc_uni:.4f}")
    print(f"Acc. Bidirezionale:  {acc_bi:.4f}")
    
    # Teoria del "Perché":
    if acc_bi > acc_uni:
        print("\nAnalisi: Il modello Bidirezionale ha vinto. Vedere il 'futuro' della")
        print("sequenza ha permesso di disambiguare pattern complessi.")
    else:
        print("\nAnalisi: Risultati simili. Su dati casuali o molto semplici,")
        print("il raddoppio dei parametri della Bidirezionale può causare Overfitting.")

# 4. ESECUZIONE DEL CONFRONTO
compare_accuracy(model_uni, model_bi, X, y, epochs=15)

# Visualizzazione dei parametri (Opzionale ma utile)
print("\n--- Confronto Complessità ---")
print(f"Parametri Modello Unidirezionale: {model_uni.count_params():,}")
print(f"Parametri Modello Bidirezionale:  {model_bi.count_params():,}")

--- Riepilogo Modello Unidirezionale ---


Model: "Unidirectional_Model"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 32)             │         5,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 5,281 (20.63 KB)

 Trainable params: 5,281 (20.63 KB)

...

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ bidirectional (Bidirectional)   │ (None, 64)             │        10,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 10,561 (41.25 KB)

 Trainable params: 10,561 (41.25 KB)

 Non-trainable params: 0 (0.00 B)


Training completato con successo.

 Inizio addestramento comparativo per 15 epoche...
Addestramento Unidirectional_Model...
Addestramento Bidirectional_Model...

RISULTATI FINALI (Dati di Validazione)
Acc. Unidirezionale: 0.4700
Acc. Bidirezionale:  0.4550

Analisi: Risultati simili. Su dati casuali o molto semplici,
il raddoppio dei parametri della Bidirezionale può causare Overfitting.

--- Confronto Complessità ---
Parametri Modello Unidirezionale: 5,281
Parametri Modello Bidirezionale:  10,561
